# Train GPT-BERT on native (non-translated) Hindi/Telugu data

Trains monolingual GPT-BERT models from scratch on the CC-100-derived native data (`pulipakav-1/hi-te`), using this project's existing `babybabellm-gptbert` training code (`Babylm2026/gpt-bert/{hindi,telugu}`).

**Trains Hindi then Telugu, one after the other, using the exact same hyperparameters as the original cluster run** (`global_batch_size=32768`, `local_batch_size=128`, `max_steps=15625` -- see `scripts/train_model.sh`). The original ran across 3-4 GPUs in parallel; on a single Colab GPU the same total compute happens sequentially instead, so expect this to take roughly 3-4x longer than the original cluster run did.

Every step below uses `subprocess.run(..., check=True)`, so a real failure **stops execution with a visible error** instead of silently continuing to the next step -- this matters because that's exactly what went wrong in an earlier version (an interrupted shard-prep step let training run anyway and fail, then the loop moved on to Telugu regardless).

In [ ]:
# Cell 1: clone the repo (idempotent -- skips cloning if BabyLM already exists, so
# re-running this cell doesn't create a nested clone inside itself) and install deps
import os

if not os.path.isdir("/content/BabyLM"):
    !git clone https://github.com/vishnup22/BabyLM.git /content/BabyLM

%cd /content/BabyLM
!git checkout evaluation
!git pull
!pip install -q -r Babylm2026/gpt-bert/hindi/requirements.txt
!pip install -q -r Babylm2026/gpt-bert/telugu/requirements.txt

REPO_ROOT = "/content/BabyLM"

In [ ]:
# Cell 2: log in to Hugging Face (only needed if you want the optional push-to-HF step at
# the end -- pulipakav-1/hi-te itself is a public dataset, no token needed just to read it)
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # use whatever secret name you saved your token under
!hf auth whoami

In [ ]:
# Cell 3: training config + the per-language pipeline as a function.
# These match Babylm2026/gpt-bert/hindi/scripts/train_model.sh EXACTLY -- the same
# hyperparameters used for the original cluster training run (which used 3-4 GPUs in
# parallel). On a single Colab GPU, the same total compute (max_steps * global_batch_size
# = ~65.5 billion tokens) has to happen sequentially instead of in parallel, so this will
# take roughly 3-4x longer than however long the original cluster run took.
import subprocess
from pathlib import Path
from huggingface_hub import hf_hub_download, HfApi

GLOBAL_BATCH_SIZE = 32768
LOCAL_BATCH_SIZE = 128  # lower this if you hit an out-of-memory error
SEQ_LENGTH = 128
MAX_STEPS = 15625


def run(cmd, cwd):
    print(f"\n$ (cwd={cwd}) {' '.join(cmd)}\n")
    subprocess.run(cmd, cwd=cwd, check=True)


def train_language(lang):
    lang_code = {"hindi": "hi", "telugu": "te"}[lang]
    dataset_name = f"native-{lang}"
    lang_dir = f"{REPO_ROOT}/Babylm2026/gpt-bert/{lang}"

    print(f"\n{'=' * 70}\nTraining {lang}\n{'=' * 70}")

    # 1. Download native data and lay it out where the tooling expects it
    src_path = hf_hub_download(repo_id="pulipakav-1/hi-te", filename=f"{lang}.txt", repo_type="dataset")
    raw_dir = Path(lang_dir) / "data" / "raw" / dataset_name
    raw_dir.mkdir(parents=True, exist_ok=True)
    dest_path = raw_dir / f"{dataset_name}.train.{lang_code}.txt"
    dest_path.write_bytes(Path(src_path).read_bytes())
    print(f"Placed {dest_path} ({dest_path.stat().st_size:,} bytes)")

    # 2. Train a fresh tokenizer on the native data (vocab_size=16384) -- not reusing the
    #    old translated-data tokenizer, which would reintroduce the tokenizer-granularity
    #    issue found earlier in this project
    run(["python", "tools/train_tokenizer_local.py",
         "--dataset", dataset_name,
         "--data_root", "data/raw",
         "--output", "tokenizers/tokenizer_native_16384.json",
         "--vocab_size", "16384"], cwd=lang_dir)

    # 3. Tokenize and shard (2% held out for validation)
    run(["python", "tools/prepare_local_shards.py",
         "--dataset", dataset_name,
         "--data_root", "data/raw",
         "--tokenizer", "tokenizers/tokenizer_native_16384.json",
         "--output_base", "data/processed",
         "--valid_fraction", "0.02"], cwd=lang_dir)

    # 4. Train, single GPU
    os.environ["WANDB_MODE"] = "disabled"  # no W&B account needed
    run(["python", "train_single_gpu.py",
         "--train_path", "../data/processed/train",
         "--valid_path", "../data/processed/valid",
         "--config_file", "../configs/base.json",
         "--tokenizer_path", "../tokenizers/tokenizer_native_16384.json",
         "--name", f"native-{lang}-gptbert",
         "--output_dir", "../model_checkpoints",
         "--hybrid_numerator", "2",
         "--hybrid_denominator", "3",
         "--global_batch_size", str(GLOBAL_BATCH_SIZE),
         "--local_batch_size", str(LOCAL_BATCH_SIZE),
         "--seq_length", str(SEQ_LENGTH),
         "--max_steps", str(MAX_STEPS),
         "--save_every", "1000",
         "--validate_every", "0",
         "--seed", "42"], cwd=f"{lang_dir}/pretraining")

    print(f"\nFinished training {lang}.")


def push_language(lang):
    lang_dir = f"{REPO_ROOT}/Babylm2026/gpt-bert/{lang}"
    repo_id = f"pulipakav-1/native-{lang}-gptbert"
    api = HfApi()
    api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
    for local_path, repo_path in [
        (f"{lang_dir}/model_checkpoints/native-{lang}-gptbert_2_3_ema.bin", "model_ema.bin"),
        (f"{lang_dir}/tokenizers/tokenizer_native_16384.json", "tokenizer.json"),
        (f"{lang_dir}/configs/base.json", "config_base.json"),
    ]:
        api.upload_file(path_or_fileobj=local_path, path_in_repo=repo_path, repo_id=repo_id, repo_type="model")
    print(f"Pushed to https://huggingface.co/{repo_id}")

In [ ]:
# Cell 4: run both languages sequentially. If a step fails, this cell will raise a real
# error and stop here -- it will NOT silently continue to the next language.
for lang in ["hindi", "telugu"]:
    train_language(lang)

## Optional: push both trained checkpoints to Hugging Face

Requires the HF login cell above to have been run with a write-scoped token.

In [ ]:
# Cell 5 (optional): push both trained checkpoints
for lang in ["hindi", "telugu"]:
    push_language(lang)